In [ ]:
import numpy as np
import random
import pickle
import hashlib
from collections import defaultdict, deque
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

# ----------------------- State–Action Utilities ------------------------
def state_to_hash(env):
    """
    Convert a normalized environment state into a hashable representation.
    Assumes env exposes normalized pieces / board info (values in [-1, 1]).
    """
    state_items = []
    for pos in sorted(env.pieces.keys()):
        piece = env.pieces[pos]
        # If piece.value etc. are already normalized, keep as is
        state_items.append((pos, piece.player, piece.value, piece.dama))
    state_tuple = (env.to_move, tuple(state_items))
    return hashlib.md5(str(state_tuple).encode()).hexdigest()[:16]


def move_to_action_id(move):
    """Convert a move to a unique action identifier."""
    start, end = move.path[0], move.path[-1]
    return f"{start[0]},{start[1]}->{end[0]},{end[1]}|cap:{len(move.captures)}"


def get_legal_actions(env):
    moves = env.generate_all_moves(env.to_move)
    return [move_to_action_id(m) for m in moves], moves


def get_legal_actions_from_state(env):
    moves = env.generate_all_moves(env.to_move)
    return [move_to_action_id(m) for m in moves], moves


# ----------------------- Normalized Dyna-Q Agent ------------------------
class DynaQAgent:
    def __init__(self, alpha=0.1, gamma=0.95, epsilon=0.1, planning_steps=50):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.planning_steps = planning_steps

        self.Q = defaultdict(lambda: defaultdict(float))
        self.model = defaultdict(lambda: defaultdict(lambda: (None, 0.0)))
        self.visited_sa = set()
        self.experience_buffer = deque(maxlen=10000)

        # track reward statistics (optional if env rewards already normalized)
        self.reward_history = deque(maxlen=1000)
        self.running_mean = 0.0
        self.running_std = 1.0

    # ---- Reward normalization (only applied if environment outputs raw rewards)
    def normalize_reward(self, r):
        if -1.0 <= r <= 1.0:
            # Already normalized by environment
            return r
        self.reward_history.append(r)
        if len(self.reward_history) > 10:
            self.running_mean = np.mean(self.reward_history)
            self.running_std = np.std(self.reward_history) + 1e-8
        z = (r - self.running_mean) / self.running_std
        return np.clip(z, -1.0, 1.0)

    # ---- ε-greedy action selection
    def get_action(self, env, legal_moves, training=True):
        state_hash = state_to_hash(env)
        action_ids, moves = get_legal_actions(env)
        if not action_ids:
            return None, None

        if training and random.random() < self.epsilon:
            chosen_action = random.choice(action_ids)
        else:
            q_values = [self.Q[state_hash][a] for a in action_ids]
            max_q = max(q_values)
            best_actions = [a for a, q in zip(action_ids, q_values) if q == max_q]
            chosen_action = random.choice(best_actions)

        chosen_move = moves[action_ids.index(chosen_action)]
        return chosen_move, chosen_action

    # ---- Q-learning update
    def update_q_value(self, state, action, reward, next_state, done):
        s = state_to_hash(state)
        if done:
            target = reward
        else:
            next_actions, _ = get_legal_actions_from_state(next_state)
            if next_actions:
                s_next = state_to_hash(next_state)
                max_next_q = max(self.Q[s_next][a] for a in next_actions)
            else:
                max_next_q = 0.0
            target = reward + self.gamma * max_next_q

        self.Q[s][action] += self.alpha * (target - self.Q[s][action])

    # ---- Model update
    def update_model(self, state, action, next_state, reward):
        s = state_to_hash(state)
        s_next = state_to_hash(next_state) if next_state else None
        self.model[s][action] = (s_next, reward)
        self.visited_sa.add((s, action))

    # ---- Planning / simulated experience
    def plan(self):
        n = min(len(self.visited_sa), self.planning_steps)
        for _ in range(n):
            s, a = random.choice(list(self.visited_sa))
            s_next, r = self.model[s][a]
            if s_next is None:
                continue
            next_actions = list(self.Q[s_next].keys())
            max_next_q = max([self.Q[s_next][a2] for a2 in next_actions], default=0.0)
            target = r + self.gamma * max_next_q
            self.Q[s][a] += self.alpha * (target - self.Q[s][a])

    # ---- Serialization
    def save_agent(self, path):
        q_dict = {s: dict(aq) for s, aq in self.Q.items()}
        model_dict = {s: dict(aq) for s, aq in self.model.items()}
        data = dict(
            Q=q_dict, model=model_dict, visited_sa=list(self.visited_sa),
            alpha=self.alpha, gamma=self.gamma, epsilon=self.epsilon,
            planning_steps=self.planning_steps,
            running_mean=self.running_mean, running_std=self.running_std,
            reward_history=list(self.reward_history)
        )
        with open(path, "wb") as f:
            pickle.dump(data, f)
        print(f"✅ Dyna-Q agent saved to {path}")

    def load_agent(self, path):
        with open(path, "rb") as f:
            d = pickle.load(f)
        self.Q = defaultdict(lambda: defaultdict(float), d["Q"])
        self.model = defaultdict(lambda: defaultdict(lambda: (None, 0.0)), d["model"])
        self.visited_sa = set(d["visited_sa"])
        self.alpha, self.gamma, self.epsilon = d["alpha"], d["gamma"], d["epsilon"]
        self.planning_steps = d["planning_steps"]
        self.running_mean, self.running_std = d.get("running_mean", 0.0), d.get("running_std", 1.0)
        self.reward_history = deque(d.get("reward_history", []), maxlen=1000)
        print(f"✅ Dyna-Q agent loaded from {path}")


# ----------------------- Training Loop ------------------------
def train_dyna_q_agent(env_factory, agent, num_episodes=500, max_moves_per_game=300):
    """
    Train Dyna-Q agent in self-play.
    Assumes env.apply_move() already returns normalized rewards in [-1, 1].
    """
    writer = SummaryWriter(f"runs/damath_dynaq_{datetime.now():%Y%m%d_%H%M%S}")
    print("📈 Training Dyna-Q with normalized environment rewards")

    win_counts = {1: 0, -1: 0, 0: 0}
    episode_rewards, episode_lengths = [], []

    for ep in range(1, num_episodes + 1):
        env = env_factory()
        total_reward, move_count = 0.0, 0

        while not env.game_over() and move_count < max_moves_per_game:
            s = env.copy()
            legal_moves = env.generate_all_moves(env.to_move)
            if not legal_moves:
                break

            move, action_id = agent.get_action(env, legal_moves, training=True)
            if move is None:
                break

            # normalized reward directly from env
            r = env.apply_move(move)
            total_reward += r

            s_next = env.copy()
            done = env.game_over()

            agent.update_q_value(s, action_id, r, s_next, done)
            agent.update_model(s, action_id, s_next, r)
            agent.plan()

            move_count += 1

        final_scores, winner = env.final_scores_and_winner()
        win_counts[winner] += 1
        episode_rewards.append(total_reward)
        episode_lengths.append(move_count)

        # simple epsilon decay
        if ep % 100 == 0:
            agent.epsilon = max(0.01, agent.epsilon * 0.95)

        # logging
        if ep % 50 == 0:
            avg_r = np.mean(episode_rewards[-50:])
            avg_len = np.mean(episode_lengths[-50:])
            winrate = win_counts[1] / ep
            writer.add_scalar("AverageReward", avg_r, ep)
            writer.add_scalar("EpisodeLength", avg_len, ep)
            writer.add_scalar("WinRatePlayer1", winrate, ep)
            writer.add_scalar("Epsilon", agent.epsilon, ep)
            print(f"Ep {ep}/{num_episodes} | R̄={avg_r:.3f} | Len={avg_len:.1f} | "
                  f"WinP1={winrate:.2f} | ε={agent.epsilon:.3f}")

    writer.close()
    print("🎯 Dyna-Q training complete.")
    return agent
